In [ ]:
import torch
from torch.nn.modules.activation import ELU
from braindecode.models import EEGNet
from torch.utils.data import DataLoader
import sys
import os
import torch
from tqdm import tqdm
from contextlib import nullcontext
from tqdm import tqdm
import copy


sys.path.append(os.path.abspath(".."))
from braindecode.util import set_random_seeds
from data.dataloader import PTStreamWindowsDataset , Loader

In [3]:
#================= Parameters =======================
# Number of EEG channels (electrodes) used as input
N_CHANS=22

# Number of output classes (for example seizure vs non-seizure, or 4 motor imagery classes)
N_OUTPUTS=14

# Number of time samples in each EEG window
# Example: 4 seconds × 250 Hz = 1000 so techniquely we should get from sfrq , window sec
N_TIMES=None

# Length of the final convolution layer
# 'auto' lets the model compute it automatically
FINAL_CONV_LENGTH='auto'

# Pooling method used in pooling layers
# Options usually: mean or max
POOL_MODE='mean'


# Number of temporal convolution filters in the first layer
# Controls how many frequency patterns the model learns
F1=8

# Depth multiplier for spatial filters
# Controls how many spatial filters are created per temporal filter
D=2

# Number of filters in the second convolution layer
# If None, it is automatically calculated as F1 * D
F2=None

# Length of the temporal convolution kernel
# Controls how much time the first filter looks at
KERNEL_LENGTH=64

# Kernel size used in depthwise convolution
DEPTHWISE_KERNEL_LENGTH=16


# Kernel size for first pooling layer
POOL1_KERNEL_SIZE=4

# Kernel size for second pooling layer
POOL2_KERNEL_SIZE=8


# Maximum norm constraint applied to spatial convolution weights
# Helps stabilize training
CONV_SPATIAL_MAX_NORM=1


# Activation function used in the network
# ELU is commonly used in EEGNet
ACTIVATION = ELU


# Momentum used in batch normalization
BATCH_NORM_MOMENTUM=0.01

# Whether batch normalization has learnable scaling parameters
BATCH_NORM_AFFINE=True

# Small value added for numerical stability in batch normalization
BATCH_NORM_EPS=0.001


# Dropout probability to prevent overfitting
DROP_PROB=0.25


# Whether to apply weight constraint in the final classification layer
FINAL_LAYER_WITH_CONSTRAINT=False


# Norm constraint applied to the classification layer weights
NORM_RATE=0.25


# Channel information (electrode names and locations)
# Usually provided when using advanced EEG datasets
CHS_INFO=None


# Length of input EEG window in seconds
INPUT_WINDOW_SECONDS=4


# Sampling frequency of EEG recording
# Example: 128 Hz, 250 Hz, 512 Hz
SFREQ=128


In [4]:
# =========== HyperParameters ============

EPOCHS = 30
LR = 1e-3

In [5]:
# model = EEGNet(
#     n_chans=N_CHANS,
#     n_outputs=N_OUTPUTS,
#     final_conv_length=FINAL_CONV_LENGTH,
#     pool_mode=POOL_MODE,
#     F1=F1,
#     D=D,
#     F2=F2,
#     kernel_length=KERNEL_LENGTH,
#     depthwise_kernel_length=DEPTHWISE_KERNEL_LENGTH,
#     pool1_kernel_size=POOL1_KERNEL_SIZE,
#     pool2_kernel_size=POOL2_KERNEL_SIZE,
#     conv_spatial_max_norm=CONV_SPATIAL_MAX_NORM,
#     activation=ACTIVATION,
#     batch_norm_momentum=BATCH_NORM_MOMENTUM,
#     batch_norm_affine=BATCH_NORM_AFFINE,
#     batch_norm_eps=BATCH_NORM_EPS,
#     drop_prob=DROP_PROB,
#     final_layer_with_constraint=FINAL_LAYER_WITH_CONSTRAINT,
#     norm_rate=NORM_RATE,
#     chs_info=CHS_INFO,
#     sfreq=SFREQ,
#     input_window_seconds=INPUT_WINDOW_SECONDS
# )

model = EEGNet(
    n_chans=41,
    n_outputs=2,
    n_times=2500,
)
cuda = torch.cuda.is_available()
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

In [6]:
#binary detection 
transform = None
d=Loader(transform=transform)
train_loader = d.return_Loader()
# batch = next(iter(train_loader))
# print(batch.keys())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

patience = 10
best_val_acc = 0
patience_counter = 0
best_model = None


def train_one_epoch(model, loader):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, leave=False)

    for batch in pbar:
        x = batch["x"].to(device, non_blocking=True)
        y = batch["y"].to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item() * y.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

        pbar.set_postfix(
            loss=f"{total_loss/total:.4f}",
            acc=f"{correct/total:.4f}"
        )

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        y = batch["y"].to(device, non_blocking=True).long()

        outputs = model(x)
        loss = criterion(outputs, y)

        total_loss += loss.item() * y.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    scheduler.step()

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | "
        f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model = copy.deepcopy(model.state_dict())
        torch.save(best_model, "best_model.pt")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping triggered")
        break


C:\Users\kakas\AppData\Local\Temp\ipykernel_22896\769183422.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
Epoch 1/30:   0%|          | 0/4176 [00:00<?, ?batch/s]C:\Users\kakas\AppData\Local\Temp\ipykernel_22896\769183422.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp_context():


Epoch [1/30] | Train Loss: 0.7009 | Train Acc: 0.7142 | LR: 1.00e-03


Epoch [2/30] | Train Loss: 0.6802 | Train Acc: 0.7281 | LR: 1.00e-03


Epoch [3/30] | Train Loss: 0.6792 | Train Acc: 0.7292 | LR: 1.00e-03


Epoch [4/30] | Train Loss: 0.6715 | Train Acc: 0.7333 | LR: 1.00e-03


...

Epoch [21/30] | Train Loss: 0.6160 | Train Acc: 0.7491 | LR: 1.00e-03


Epoch [22/30] | Train Loss: 0.6123 | Train Acc: 0.7504 | LR: 1.00e-03


Epoch [23/30] | Train Loss: 0.6119 | Train Acc: 0.7508 | LR: 1.00e-03


Epoch [24/30] | Train Loss: 0.6063 | Train Acc: 0.7521 | LR: 1.00e-03


KeyboardInterrupt: 